# Gán nhãn dữ liệu tin tuyển dụng CNTT (salary-v2)

Notebook trình bày từng bước biến **tin tuyển dụng** thô thành bộ dữ liệu có **nhãn lương** và nhãn đặc trưng (vai trò, cấp bậc, kinh nghiệm, kỹ năng), rồi đo độ chính xác của bộ gán nhãn bằng mẫu duyệt.

- Mọi luật gán nhãn nằm trong package `career_market`; notebook chỉ **gọi lại** và minh họa, không viết lại logic. Sửa luật thì sửa ở package, notebook tự phản ánh.
- Notebook được commit **không có output**: bảng và biểu đồ sinh ra khi chạy.
- Bước bắt buộc trước khi huấn luyện lại vẫn là lệnh `python -m career_market build-dataset` cùng bộ test; notebook để trình bày và kiểm tra. CI chạy thử notebook để nó không bị hỏng.

Chạy: mở bằng Jupyter/VS Code với môi trường `job_market/.venv`, chọn *Run All*.

In [ ]:
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from career_market.label_review import score_review
from career_market.salary_dataset import (
    build_dataset,
    classify_role,
    data_root,
    derive_level,
    extract_skills,
    load_dataset,
    parse_experience,
    parse_salary,
    skill_lists,
)

pd.set_option("display.max_colwidth", 80)
RAW = data_root() / "raw"
saved = load_dataset()
print("Dữ liệu thô:", sorted(p.name for p in RAW.glob("*.csv")))
print("Bộ dữ liệu đã lưu:", saved.shape)

## 1. Đọc dữ liệu thô và phễu lọc

`build_dataset` đọc ba nguồn (IT Jobs 2026, TopCV 2026, CareerBuilder 2023), giữ tin có lương đọc được, giữ tin thuộc **vai trò nghề nghiệp** CNTT, rồi bỏ tin trùng. Phễu cho biết mỗi bước loại bao nhiêu tin.

In [ ]:
frame, funnel = build_dataset(RAW)
steps = pd.Series(funnel, name="Số tin").rename(
    {
        "raw": "Tin thô",
        "has_salary": "Có lương đọc được",
        "it_role": "Thuộc vai trò CNTT",
        "deduplicated": "Sau bỏ trùng",
    }
)
display(steps.to_frame())
ax = steps.plot.barh(color="#2563eb", figsize=(6, 2.5))
ax.invert_yaxis()
ax.set_xlabel("Số tin")
plt.show()

# Dựng lại phải ra đúng bộ dữ liệu đã lưu, để nhãn tái lập được.
assert len(frame) == len(saved)
assert frame["salary"].round(3).equals(saved["salary"].round(3))
frame.groupby("source").size().rename("Số tin có nhãn").to_frame()

## 2. Nhãn lương

`parse_salary` chuẩn hóa chuỗi lương về `(thấp, cao, trung điểm)` triệu VND/tháng: đổi USD theo tỉ giá năm đăng, chia 12 với lương năm, bỏ tin "thỏa thuận". Nhãn là **lương trung điểm**.

Hai luật tạo nhiễu cần biết (xem #77):
- tin chỉ ghi một đầu: "Tới X" → thấp = 0.7·X; "Từ X" → cao = 1.3·X;
- khoảng có cao/thấp > 3 bị loại vì không mang thông tin.

In [ ]:
examples = [
    "15 - 25 triệu",
    "Lương: 10 Tr - 15 Tr VND",
    "$ 1,200-1,500 /tháng",
    "240tr-360tr ₫/năm",
    "Tới 30 triệu",
    "Từ 20 triệu",
    "Thoả thuận",
    "1 triệu - 40 triệu/tháng",
]
pd.DataFrame(
    [{"Chuỗi lương": text, "Thấp–cao–trung điểm": parse_salary(text)} for text in examples]
)

In [ ]:
ratio = frame["salary_max"] / frame["salary_min"]
one_sided = frame["salary_text"].str.contains(r"tới|đến|up to|từ|trên", case=False, regex=True)
print(f"Tin chỉ ghi một đầu lương: {one_sided.sum()} ({one_sided.mean():.1%})")
print(f"Cao/thấp: trung vị {ratio.median():.2f}, p90 {ratio.quantile(0.9):.2f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
frame["salary"].plot.hist(bins=40, ax=axes[0], color="#2563eb")
axes[0].set(title="Lương trung điểm (triệu/tháng)", ylabel="Số tin")
ratio.plot.hist(bins=30, ax=axes[1], color="#64748b")
axes[1].set(title="Độ rộng khoảng (cao/thấp)", ylabel="Số tin")
plt.show()

## 3. Vai trò, cấp bậc, kinh nghiệm, kỹ năng

- `classify_role`: chức danh → một trong 23 vai trò; trả `None` nếu ngoài ngành (tin bị loại).
- `parse_experience`: chuỗi kinh nghiệm → số năm; "Không yêu cầu" = 0; khoảng "1 - 5 năm" lấy giữa.
- `derive_level`: ưu tiên từ khóa trong chức danh/cấp bậc nguồn, nếu không thì suy từ số năm.
- `extract_skills`: khớp danh mục kỹ năng trên chức danh và mô tả.

In [ ]:
titles = [
    "Senior Backend Engineer",
    "MERN Stack Developer",
    "QA Tech Lead",
    "Nhân viên kinh doanh",
    "Product Manager",
    "Data Scientist",
]
pd.DataFrame(
    [
        {"Chức danh": t, "Vai trò": classify_role(t), "Cấp bậc (2 năm)": derive_level(t, 2)}
        for t in titles
    ]
)

In [ ]:
texts = ["Không yêu cầu", "Dưới 1 năm", "Trên 2 Năm", "1 - 5 Năm", "3 năm", "Không hiển thị"]
display(pd.DataFrame([{"Chuỗi kinh nghiệm": t, "Số năm": parse_experience(t)} for t in texts]))
jd = "Yêu cầu: thành thạo Python, SQL; có kinh nghiệm Docker, AWS là lợi thế. API Testing với Postman."
print("Kỹ năng trích từ đoạn JD:", extract_skills(jd))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
frame["role"].value_counts().sort_values().plot.barh(ax=axes[0], color="#2563eb")
axes[0].set(title="Số tin theo vai trò", xlabel="Số tin")
skills = skill_lists(frame).explode().dropna().value_counts().head(20).sort_values()
skills.plot.barh(ax=axes[1], color="#64748b")
axes[1].set(title="20 kỹ năng được nhắc nhiều nhất", xlabel="Số tin")
plt.tight_layout()
plt.show()
frame["level"].value_counts().rename("Số tin").to_frame().T

## 4. Bỏ trùng và chia train/valid/test theo công ty

Tin trùng (cùng chức danh chuẩn hóa, công ty và lương) chỉ giữ một bản. Mỗi công ty nằm trọn trong một tập (ADR 0008), nên đánh giá là dự đoán cho **công ty chưa gặp**.

In [ ]:
print("Bị loại do trùng:", funnel["it_role"] - funnel["deduplicated"])
display(frame["split"].value_counts().rename("Số tin").to_frame().T)
companies_per_split = frame.groupby("company_key")["split"].nunique()
assert (companies_per_split == 1).all(), "Một công ty không được nằm ở hai tập"
print("Số công ty:", frame["company_key"].nunique(), "· không công ty nào nằm ở hai tập")

## 5. Độ chính xác của bộ gán nhãn

Mẫu 208 tin (phân tầng theo nguồn, rải đều vai trò) được duyệt ở #79. Lưu ý: mẫu do **AI duyệt**, chưa phải nhãn chuẩn của người duyệt. Muốn duyệt lại: tạo mẫu bằng `career_market.label_review.review_sample()`, điền các cột `review_*`, rồi chạy `score_review`.

In [ ]:
review = pd.read_csv(ROOT / "datasets/labels/2026-11-06/label_accuracy_review.csv", dtype=str)
scores = score_review(review)
display(
    pd.DataFrame(
        [
            {
                "Nhãn": k,
                "Số dòng chấm": v["reviewed"],
                "Tỉ lệ đúng (%)": round(v["accuracy_pct"], 1),
            }
            for k, v in scores.items()
            if k != "skills"
        ]
        + [
            {
                "Nhãn": "skills (độ chính xác)",
                "Số dòng chấm": scores["skills"]["assigned"],
                "Tỉ lệ đúng (%)": round(scores["skills"]["precision_pct"], 1),
            }
        ]
    )
)
review.loc[review["review_note"].notna(), ["title", "role", "level", "review_note"]].head(15)

## 6. Xuất bộ dữ liệu

Bản chính thức được ghi bằng `python -m career_market build-dataset`. Ở đây xuất ra thư mục tạm và đối chiếu với bản đã lưu, để chắc chắn notebook và lệnh xuất cho cùng một kết quả.

In [ ]:
from career_market.salary_dataset import write_dataset

output = Path(tempfile.mkdtemp()) / "salary_dataset_v2.csv"
write_dataset(RAW, output)
exported = pd.read_csv(output)
assert exported.shape == saved.shape
assert (exported["row_id"] == saved["row_id"]).all()
print("Đã xuất", exported.shape, "→ khớp bộ dữ liệu đã lưu")